<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Deep Learning on Images — Optional
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

## How to use this notebook

**Optional, about 45 minutes, nothing to submit.** Session 6's section on deep learning ended with "where depth pays": images. Here a convolutional neural network (CNN) learns to name the organ in a slice of an abdominal CT scan, and you compare it with the `MLPClassifier` you met in the session, given the same pixels.

Most of the code is written for you. The tasks are the ones you know: **» Predict** before running, **» Investigate** a working cell, and **» Modify** one thing.

* **Runtime → Change runtime type → T4 GPU** makes the network train in seconds rather than minutes. It works without one.
* The images are public, from [MedMNIST](https://medmnist.com/), under a [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) licence, so you may plot them and share your outputs. They are not for clinical use. MIMIC rules still apply to every other notebook.
* As in the session, every choice is made on the **validation** images, and the **test** images are used once, at the end.

## Setup

In [ ]:
%pip install medmnist --quiet
import numpy as np
import matplotlib.pyplot as plt
import keras
from medmnist import OrganAMNIST

train = OrganAMNIST(split='train', download=True, size=28)
val = OrganAMNIST(split='val', download=True, size=28)
test = OrganAMNIST(split='test', download=True, size=28)

# Pixels scaled from 0-255 to 0-1, with one channel: (images, height, width, 1)
X_train = train.imgs[..., np.newaxis] / 255
X_val = val.imgs[..., np.newaxis] / 255
X_test = test.imgs[..., np.newaxis] / 255

# The organ in each image, as a number from 0 to 10
y_train = train.labels.ravel()
y_val = val.labels.ravel()
y_test = test.labels.ravel()

organs = [train.info['label'][str(number)] for number in range(11)]
print(X_train.shape, X_val.shape, X_test.shape)
print(organs)

## 1. Look at the data
One training image of each organ.

In [ ]:
fig, axes = plt.subplots(1, 11, figsize=(16, 2))
for number, ax in enumerate(axes):
    first = np.where(y_train == number)[0][0]
    ax.imshow(X_train[first, :, :, 0], cmap='gray')
    ax.set_title(organs[number], fontsize=8)
    ax.axis('off')
plt.show()

**» Investigate** — Each image is one slice of a CT scan, shrunk to 28 × 28 pixels. **Which two organs would you find hardest to tell apart at this size?** Keep your answer: section 4 checks it.

## 2. The network you know
`MLPClassifier` takes a table, so each image becomes one row of 784 numbers, its pixels in order. Every pixel is a separate feature: the network is not told which pixels are next to each other.

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier

# Each 28 x 28 image becomes one row of 784 pixels
rows_train = X_train.reshape(len(X_train), -1)
rows_val = X_val.reshape(len(X_val), -1)

mlp = MLPClassifier(hidden_layer_sizes=(100,), early_stopping=True, random_state=0)
mlp.fit(rows_train, y_train)

pred_mlp = mlp.predict(rows_val)
print('MLP, validation accuracy:', round(accuracy_score(y_val, pred_mlp), 3))

## 3. A convolutional network
*   A **convolutional layer** slides small filters, 3 × 3 pixels here, across the image. Each filter learns to respond to one pattern, such as an edge, wherever it appears
*   A **pooling layer** keeps the strongest response in each 2 × 2 patch, halving the image's width and height
*   After two rounds of both, `Flatten` and the `Dense` layers are the network from the session
*   The output layer has **11 neurons, one per organ**, and the **softmax** turns them into 11 probabilities that add up to 1 (see the session's appendix)
*   `fit` passes through the training images several times, one **epoch** each. **Early stopping** watches the validation accuracy after every epoch, stops when it has not improved for three, and keeps the best epoch's weights: the session's `early_stopping`, for Keras

**» Predict** — The MLP's validation accuracy is above. **Write down the validation accuracy you expect from the CNN.**

In [ ]:
# Fixes the random starting weights, so a re-run gives the same network
keras.utils.set_random_seed(0)

cnn = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(16, 3, activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 3, activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(11, activation='softmax'),
])
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Stop when validation accuracy has not improved for 3 epochs, and keep the best epoch
stop = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
cnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=128, callbacks=[stop], verbose=2)

prob_cnn = cnn.predict(X_val, verbose=0)
pred_cnn = prob_cnn.argmax(axis=1)
print('CNN, validation accuracy:', round(accuracy_score(y_val, pred_cnn), 3))

**» Investigate** — **Which network has more weights, and by how much?** The CNN beat the MLP: was it by having more?

In [ ]:
mlp_weights = sum(layer.size for layer in mlp.coefs_) + sum(layer.size for layer in mlp.intercepts_)
print('MLP weights:', mlp_weights)
print('CNN weights:', cnn.count_params())

## 4. What it gets wrong
Each row is the true organ, each column the CNN's answer, on the validation images.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(y_val, pred_cnn, display_labels=organs, xticks_rotation=90, colorbar=False, ax=ax)
plt.show()

**» Investigate** — **Which pairs of organs does the CNN confuse most?** Are they the ones you predicted in section 1? Why might those be hard?

## 5. Holding it back: dropout
The session's networks were held back with `alpha`. A common way for image networks is **dropout**: at every training step it switches off a random half of the neurons in the layer before it, so the network cannot lean on any one of them. It is switched off when predicting.

**» Modify** — **Add the line in the comment straight before `keras.layers.Dense(64, ...)`**, run the cell, and compare the validation accuracy with the CNN's above. Early stopping may now run all 15 epochs: why might that be?

In [ ]:
# keras.layers.Dropout(0.5),

keras.utils.set_random_seed(0)

cnn_dropout = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(16, 3, activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 3, activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(11, activation='softmax'),
])
cnn_dropout.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

stop = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
cnn_dropout.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=128, callbacks=[stop], verbose=2)

pred_dropout = cnn_dropout.predict(X_val, verbose=0).argmax(axis=1)
print('CNN with dropout, validation accuracy:', round(accuracy_score(y_val, pred_dropout), 3))

## 6. The test, once
Your best network on validation, and the MLP, each tested once on the test images. Change `best` to `cnn_dropout` if dropout did better on validation.

In [ ]:
best = cnn

pred_test_cnn = best.predict(X_test, verbose=0).argmax(axis=1)
rows_test = X_test.reshape(len(X_test), -1)
pred_test_mlp = mlp.predict(rows_test)

print('MLP, test accuracy:', round(accuracy_score(y_test, pred_test_mlp), 3))
print('CNN, test accuracy:', round(accuracy_score(y_test, pred_test_cnn), 3))

**» Investigate** — **Compare each network's test accuracy with its validation accuracy.** The training and validation images come from 131 CT scans, 16 of them held back for validation; the test images come from a separate set of 70 scans. What does any gap tell you about a model moving to a new hospital's scanners?

## To think about
1. On the session's table of 25 columns, a network barely beat logistic regression. **What did the convolutions buy here, and why could they not help on the table?**
2. This network names the organ in a slice it is shown. **What would a hospital need to check before using a model like this on its own scans?**

### Sources
*   Yang J, Shi R, Wei D, et al. MedMNIST v2: a large-scale lightweight benchmark for 2D and 3D biomedical image classification. *Scientific Data* 2023;10:41. [doi:10.1038/s41597-022-01721-8](https://doi.org/10.1038/s41597-022-01721-8)
*   OrganAMNIST: axial slices through 11 organs, from the CT scans of the Liver Tumor Segmentation Benchmark (LiTS), described in the MedMNIST v2 paper.